# Python Iterators & Generators — Complete Advanced Guide (with AI/ML Engineering Applications)

Iterators and generators are two sides of the same coin: a generator is simply the *easiest way* to build something that follows the iterator protocol. This notebook covers both together, then builds up to the exact streaming/lazy-loading patterns used in real ML data pipelines (PyTorch `DataLoader`/`IterableDataset`, `tf.data`, NLP tokenization streams, etc).

**Table of Contents**

**Part 1 — Iterators**
1. Iterable vs Iterator
2. `iter()` and `next()`
3. The Iterator Protocol — Building a Custom Iterator
4. `StopIteration` and How `for` Loops Really Work
5. Infinite Iterators & the `itertools` Toolbox

**Part 2 — Generators**
6. Generator Functions — `yield` Basics
7. Generators Are Iterators
8. Generator Expressions vs List Comprehensions
9. Lazy Evaluation — Practical: Reading Large Files
10. Advanced Generator Methods: `send()`, `throw()`, `close()`
11. `yield from` — Generator Delegation
12. Generator Pipelines

**Part 3 — AI/ML Applications**
13. Building a Mini Batch DataLoader from Scratch
14. Lazy/Streaming Preprocessing Pipeline
15. Custom `IterableDataset`-Style Class (PyTorch Pattern)
16. Infinite Data-Augmentation Generator for Training Loops
17. Sliding-Window / Sequence-Chunking Generator (NLP & Time-Series)
18. Memory & Performance: List vs Generator at Scale

**Part 4 — Wrap Up**
19. Pitfalls & Best Practices
20. Cheat Sheet

## 1. Iterable vs Iterator

These two words are often used interchangeably, but they mean different things:

- An **iterable** is anything you can call `iter()` on — it implements `__iter__`. Lists, strings, dicts, files, sets are all iterables.
- An **iterator** is the stateful object `iter()` *produces* — it implements both `__iter__` and `__next__`, and remembers where it is between calls.

A list is iterable, but it is **not** itself an iterator — it has no memory of "where you are" in a loop. That memory lives in the iterator object that `iter()` creates from it.

In [1]:
my_list = [1, 2, 3]

print("List has __iter__:", hasattr(my_list, "__iter__"))   # True  -> iterable
print("List has __next__:", hasattr(my_list, "__next__"))   # False -> NOT an iterator itself

my_iterator = iter(my_list)
print("Iterator has __iter__:", hasattr(my_iterator, "__iter__"))  # True
print("Iterator has __next__:", hasattr(my_iterator, "__next__"))  # True -> this IS an iterator

print(type(my_list))
print(type(my_iterator))

List has __iter__: True
List has __next__: False
Iterator has __iter__: True
Iterator has __next__: True
<class 'list'>
<class 'list_iterator'>


## 2. `iter()` and `next()`

`iter(obj)` calls `obj.__iter__()` to get an iterator. `next(it)` calls `it.__next__()` to advance it by one step. Once an iterator is exhausted, every further call to `next()` raises `StopIteration`.

In [2]:
my_list = [1, 2, 3, 4, 5]
Iterator = iter(my_list)
print(type(Iterator))

print(next(Iterator))
print(next(Iterator))
print(next(Iterator))
print(next(Iterator))
print(next(Iterator))

try:
    next(Iterator)
except StopIteration:
    print("There are no elements in the Iterator")

<class 'list_iterator'>
1
2
3
4
5
There are no elements in the Iterator


In [3]:
# next() accepts a default value, returned instead of raising once exhausted
iterator2 = iter([1, 2])
print(next(iterator2, "no more items"))
print(next(iterator2, "no more items"))
print(next(iterator2, "no more items"))   # exhausted -> default returned, no exception

1
2
no more items


In [4]:
# Strings are iterable too
my_string = "Helllo"
string_iterator = iter(my_string)

print(next(string_iterator))
print(next(string_iterator))

H
e


## 3. The Iterator Protocol — Building a Custom Iterator

Any class that implements **both** `__iter__` (returning `self`) and `__next__` (returning the next value, or raising `StopIteration` when done) is a valid iterator. It will work seamlessly with `for` loops, `list()`, `sum()`, and every other construct that expects an iterable.

In [5]:
class CountUpTo:
    """A custom iterator that counts from 1 up to `limit`."""
    def __init__(self, limit):
        self.limit = limit
        self.counter = 0

    def __iter__(self):
        return self          # an iterator's __iter__ must return itself

    def __next__(self):
        if self.counter >= self.limit:
            raise StopIteration
        self.counter += 1
        return self.counter

for num in CountUpTo(5):
    print(num)

1
2
3
4
5


Now the same protocol applied to something ML-shaped: a from-scratch **batch iterator**, the core idea behind every `DataLoader`.

In [6]:
class BatchIterator:
    """Iterates over a dataset in fixed-size batches -- the core idea behind any DataLoader."""
    def __init__(self, data, batch_size):
        self.data = data
        self.batch_size = batch_size
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.data):
            raise StopIteration
        batch = self.data[self.index : self.index + self.batch_size]
        self.index += self.batch_size
        return batch

dataset = list(range(10))     # pretend these are 10 training samples
for batch in BatchIterator(dataset, batch_size=3):
    print(batch)

[0, 1, 2]
[3, 4, 5]
[6, 7, 8]
[9]


## 4. `StopIteration` and How `for` Loops Really Work

A `for` loop is just syntactic sugar. `for x in iterable:` is roughly equivalent to:

```python
it = iter(iterable)
while True:
    try:
        x = next(it)
    except StopIteration:
        break
    # loop body
```

Understanding this is what makes generators (next section) click — a generator function is just a very convenient way to write the body of `__next__` without writing the surrounding class at all.

In [7]:
def manual_for_loop(iterable):
    """Reimplements what 'for x in iterable:' does internally."""
    it = iter(iterable)
    while True:
        try:
            item = next(it)
        except StopIteration:
            break
        print(item)

manual_for_loop([10, 20, 30])

10
20
30


## 5. Infinite Iterators & the `itertools` Toolbox

Iterators don't have to be finite. The standard library `itertools` module ships several infinite iterators (`count`, `cycle`, `repeat`) plus tools to safely bound or combine them (`islice`, `chain`).

In [8]:
import itertools

# count() -- infinite counter, useful for assigning incrementing step/epoch numbers
step_counter = itertools.count(start=1, step=1)
for _ in range(5):
    print(next(step_counter))

1
2
3
4
5


In [9]:
# cycle() -- infinite repeat of a sequence, e.g. round-robin assignment across GPUs
devices = itertools.cycle(["cuda:0", "cuda:1"])
for _ in range(5):
    print(next(devices))

cuda:0
cuda:1
cuda:0
cuda:1
cuda:0


In [10]:
# islice() -- safely take a finite slice out of an infinite iterator/generator
first_10_squares = itertools.islice((i ** 2 for i in itertools.count()), 10)
print(list(first_10_squares))

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


In [11]:
# chain() -- iterate over several iterables back-to-back without copying them into one list
train_files = ["train_part1.csv", "train_part2.csv"]
val_files = ["val_part1.csv"]

for f in itertools.chain(train_files, val_files):
    print(f)

train_part1.csv
train_part2.csv
val_part1.csv


## 6. Generator Functions — `yield` Basics

A **generator function** is any function containing `yield`. Calling it doesn't run the body at all — it instantly returns a *generator object*. Each call to `next()` runs the body until the next `yield`, then pauses, remembering all local state, until `next()` is called again.

This is the "simpler way to create iterators" mentioned at the very start: it gives you the full iterator protocol from Section 3 *for free*, with no class, no manual `self.counter`, no manual `StopIteration`.

In [12]:
def square(n):
    """Yield the squares of 0..n-1, one at a time."""
    for i in range(n):
        yield i ** 2

gen = square(5)
print(gen)                  # <generator object square at ...> -- nothing has run yet!

for value in gen:
    print(value)

<generator object square at 0x7fb48f780790>
0
1
4
9
16


In [13]:
def my_generator():
    print("Starting")
    yield 1
    print("Resumed after first yield")
    yield 2
    print("Resumed after second yield")
    yield 3
    print("Generator function body finished")

gen = my_generator()
print(next(gen))
print(next(gen))
print(next(gen))
try:
    next(gen)
except StopIteration:
    print("Generator exhausted")

Starting
1
Resumed after first yield
2
Resumed after second yield
3
Generator function body finished
Generator exhausted


Notice the print statements appear *interleaved* with the returned values — proof that the function body genuinely pauses and resumes, rather than running to completion up front.

## 7. Generators Are Iterators

A generator object automatically has both `__iter__` and `__next__` — it satisfies the exact protocol from Section 3. This is why writing your own iterator class by hand (like `CountUpTo`) is rarely necessary in practice; a generator function does the same job with far less code.

In [14]:
def square(n):
    for i in range(n):
        yield i ** 2

gen = square(3)
print("Has __iter__:", hasattr(gen, "__iter__"))
print("Has __next__:", hasattr(gen, "__next__"))
print("iter(gen) is gen:", iter(gen) is gen)   # a generator's __iter__ returns itself, just like CountUpTo did

Has __iter__: True
Has __next__: True
iter(gen) is gen: True


## 8. Generator Expressions vs List Comprehensions

A **generator expression** uses parentheses instead of brackets and behaves exactly like a generator function: it builds nothing up front, and produces values lazily, one at a time.

In [15]:
import sys

list_comp = [x ** 2 for x in range(100_000)]      # built eagerly, fully materialized in memory
gen_exp   = (x ** 2 for x in range(100_000))       # built lazily, almost no memory used

print(f"List comprehension size:   {sys.getsizeof(list_comp):,} bytes")
print(f"Generator expression size: {sys.getsizeof(gen_exp):,} bytes")

print(sum(list_comp))
print(sum(gen_exp))     # works identically with sum(), any(), max(), etc.

List comprehension size:   800,984 bytes
Generator expression size: 200 bytes
333328333350000
333328333350000


## 9. Lazy Evaluation — Practical: Reading Large Files

Generators are particularly useful for reading large files because they let you process one line at a time without ever loading the entire file into memory. This is exactly the technique behind streaming massive text corpora, log files, or CSV datasets that are far larger than available RAM.

(We create a small sample file here so the example is fully self-contained and runnable.)

In [16]:
# Create a sample multi-line file to read -- a stand-in for a much larger dataset/corpus file
sample_lines = [
    "Generators yield values lazily, one at a time.\n",
    "This keeps memory usage flat regardless of file size.\n",
    "Each call to next() resumes the function exactly where it left off.\n",
    "This is essential when training on corpora too large to fit in RAM.\n",
]
with open("sample_corpus.txt", "w") as f:
    f.writelines(sample_lines)

In [17]:
def read_large_file(file_path):
    """Yield one line at a time -- memory usage stays constant no matter the file size."""
    with open(file_path, "r") as file:
        for line in file:
            yield line.strip()

for line in read_large_file("sample_corpus.txt"):
    print(line)

Generators yield values lazily, one at a time.
This keeps memory usage flat regardless of file size.
Each call to next() resumes the function exactly where it left off.
This is essential when training on corpora too large to fit in RAM.


In [18]:
def word_count_stream(file_path):
    """Lazily yields the word count of each line -- never holds the whole file in memory."""
    for line in read_large_file(file_path):
        yield len(line.split())

total_words = sum(word_count_stream("sample_corpus.txt"))
print(f"Total words across the file: {total_words}")

Total words across the file: 42


In [19]:
import os
os.remove("sample_corpus.txt")   # tidy up the sample file

## 10. Advanced Generator Methods: `send()`, `throw()`, `close()`

`yield` is actually an *expression*, not just a statement — it can both produce a value and receive one. `generator.send(value)` resumes the generator, making `value` the result of the paused `yield` expression. `generator.throw(exc)` raises an exception at the paused `yield` point. `generator.close()` stops the generator early by raising `GeneratorExit` inside it.

This makes generators genuinely coroutine-like — useful for things like tracking a running average of loss/accuracy without storing the entire history.

In [20]:
def running_average():
    """Receives new values via send() and yields the running average after each one."""
    total = 0.0
    count = 0
    average = None
    while True:
        value = yield average      # pause here; .send(x) resumes with value = x
        total += value
        count += 1
        average = total / count

avg_gen = running_average()
next(avg_gen)                 # "prime" the generator -- advance it to the first yield

print(avg_gen.send(10))       # running average so far: 10.0
print(avg_gen.send(20))       # running average so far: 15.0
print(avg_gen.send(30))       # running average so far: 20.0

10.0


15.0
20.0


In [21]:
def safe_resource_generator():
    try:
        print("Resource acquired")
        yield 1
        yield 2
    except GeneratorExit:
        print("Generator closed early -- cleaning up resource")
    finally:
        print("Cleanup complete")

gen = safe_resource_generator()
print(next(gen))
gen.close()        # triggers GeneratorExit inside the generator, runs the cleanup code

Resource acquired
1
Generator closed early -- cleaning up resource
Cleanup complete


## 11. `yield from` — Generator Delegation

`yield from sub_iterable` delegates iteration to another generator (or any iterable), forwarding every value it produces — and, in coroutine use, forwarding `send()`/`throw()` calls too. It flattens nested generator calls cleanly.

In [22]:
def inner_generator():
    yield 1
    yield 2
    yield 3

def outer_generator():
    yield "start"
    yield from inner_generator()   # delegates to inner_generator, forwarding each value
    yield "end"

print(list(outer_generator()))

['start', 1, 2, 3, 'end']


ML-shaped version: flattening several dataset **shards** into one continuous stream — exactly what you'd do when a dataset is split across multiple files/workers.

In [23]:
def stream_dataset_shard(shard_id, num_items=3):
    for i in range(num_items):
        yield f"shard{shard_id}-item{i}"

def stream_all_shards(num_shards):
    for shard_id in range(num_shards):
        yield from stream_dataset_shard(shard_id)   # flatten multiple shards into one stream

print(list(stream_all_shards(3)))

['shard0-item0', 'shard0-item1', 'shard0-item2', 'shard1-item0', 'shard1-item1', 'shard1-item2', 'shard2-item0', 'shard2-item1', 'shard2-item2']

## 12. Generator Pipelines

Chaining generators together builds a lazy "pipeline" — each stage processes one item at a time and passes it downstream, never materializing a full intermediate list. This exact idea underlies `tf.data` pipelines and PyTorch `IterableDataset` chains.

In [24]:
def read_numbers():
    for i in range(1, 11):
        yield i

def filter_even(numbers):
    for n in numbers:
        if n % 2 == 0:
            yield n

def square_all(numbers):
    for n in numbers:
        yield n ** 2

pipeline = square_all(filter_even(read_numbers()))
print(list(pipeline))     # nothing is computed until we actually consume the pipeline

[4, 16, 36, 64, 100]

In [25]:
# The same pipeline, built from built-in lazy tools instead of custom generators
pipeline2 = map(lambda n: n ** 2, filter(lambda n: n % 2 == 0, range(1, 11)))
print(list(pipeline2))

[4, 16, 36, 64, 100]


## 13. Building a Mini Batch DataLoader from Scratch

Combine a generator with shuffling and batching, and you have re-implemented the essence of `torch.utils.data.DataLoader` — in plain Python, with no dependencies.

In [26]:
import random

def data_loader(dataset, batch_size=4, shuffle=True):
    """A minimal, from-scratch version of what DataLoader does under the hood."""
    indices = list(range(len(dataset)))
    if shuffle:
        random.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start : start + batch_size]
        yield [dataset[i] for i in batch_indices]

dataset = [f"sample_{i}" for i in range(10)]

random.seed(42)
for batch in data_loader(dataset, batch_size=3):
    print(batch)

['sample_7', 'sample_3', 'sample_2']
['sample_8', 'sample_5', 'sample_6']
['sample_9', 'sample_4', 'sample_0']
['sample_1']


## 14. Lazy/Streaming Preprocessing Pipeline

A typical NLP-style preprocessing pipeline — clean → tokenize → batch — built entirely from chained generators, so raw text is never fully duplicated in memory at any single stage.

In [27]:
raw_texts = [
    "Generators make data pipelines memory-efficient.",
    "",
    "Lazy evaluation avoids loading everything at once.",
    "  ",
    "Stream large corpora without exhausting RAM.",
]

def clean_stream(texts):
    for t in texts:
        t = t.strip()
        if t:                       # drop empty / whitespace-only lines lazily
            yield t

def tokenize_stream(texts):
    for t in texts:
        yield t.lower().split()

def batch_stream(tokenized, batch_size=2):
    batch = []
    for item in tokenized:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:                        # don't forget the final partial batch
        yield batch

pipeline = batch_stream(tokenize_stream(clean_stream(raw_texts)), batch_size=2)
for batch in pipeline:
    print(batch)

[['generators', 'make', 'data', 'pipelines', 'memory-efficient.'], ['lazy', 'evaluation', 'avoids', 'loading', 'everything', 'at', 'once.']]


[['stream', 'large', 'corpora', 'without', 'exhausting', 'ram.']]


## 15. Custom `IterableDataset`-Style Class (PyTorch Pattern)

`torch.utils.data.IterableDataset` expects you to implement `__iter__`, and that method is almost always implemented as a generator. This is exactly the iterator/generator protocol from Parts 1 & 2, applied directly inside a real framework's API. Here's a dependency-free re-implementation of the same shape.

In [28]:
class StreamingTextDataset:
    """Mirrors the shape of torch.utils.data.IterableDataset: __iter__ returns a generator."""
    def __init__(self, lines):
        self.lines = lines     # in real use this would be a file path / network stream

    def __iter__(self):
        for line in self.lines:
            line = line.strip()
            if line:
                yield {"text": line, "length": len(line)}

raw_lines = [
    "First training example.\n",
    "\n",
    "Second training example, slightly longer.\n",
]

dataset = StreamingTextDataset(raw_lines)
for example in dataset:
    print(example)

{'text': 'First training example.', 'length': 23}
{'text': 'Second training example, slightly longer.', 'length': 41}


## 16. Infinite Data-Augmentation Generator for Training Loops

Training loops are usually run for a fixed number of *steps*, not a fixed number of dataset passes. An infinite generator that cycles the dataset — reshuffling each epoch and applying augmentation on the fly — is the standard pattern for feeding such a loop.

In [29]:
import itertools
import random

def augment(sample):
    """Pretend augmentation -- e.g. a random flip/crop/noise injection on an image sample."""
    return f"{sample}(augmented:{random.randint(0, 999)})"

def infinite_augmented_stream(dataset, shuffle_each_epoch=True):
    while True:                                   # loop forever across epochs
        epoch_data = dataset[:]
        if shuffle_each_epoch:
            random.shuffle(epoch_data)
        for sample in epoch_data:
            yield augment(sample)

dataset = ["img_1", "img_2", "img_3"]
random.seed(0)

stream = infinite_augmented_stream(dataset)
for step, sample in enumerate(itertools.islice(stream, 7)):   # simulate exactly 7 training steps
    print(f"step {step}: {sample}")

step 0: img_1(augmented:41)
step 1: img_3(augmented:265)
step 2: img_2(augmented:988)
step 3: img_1(augmented:414)
step 4: img_2(augmented:940)
step 5: img_3(augmented:802)
step 6: img_1(augmented:366)


## 17. Sliding-Window / Sequence-Chunking Generator (NLP & Time-Series)

Many ML tasks need fixed-size, often overlapping windows over a sequence — n-gram/context windows for language modeling, or sliding windows over a time series for forecasting. A generator computes these lazily, without ever materializing every window in memory at once.

In [30]:
def sliding_window(sequence, window_size, step=1):
    """Yield successive overlapping windows of `window_size` over `sequence`."""
    for start in range(0, len(sequence) - window_size + 1, step):
        yield sequence[start : start + window_size]

tokens = ["the", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]

for window in sliding_window(tokens, window_size=3, step=1):
    print(window)

['the', 'quick', 'brown']
['quick', 'brown', 'fox']
['brown', 'fox', 'jumps']
['fox', 'jumps', 'over']
['jumps', 'over', 'the']
['over', 'the', 'lazy']
['the', 'lazy', 'dog']


In [31]:
# The same generator applied to a time-series, e.g. computing a moving average
prices = [100, 101, 99, 105, 110, 108, 112]

for window in sliding_window(prices, window_size=3):
    avg = sum(window) / len(window)
    print(f"window={window} -> moving_avg={avg:.2f}")

window=[100, 101, 99] -> moving_avg=100.00
window=[101, 99, 105] -> moving_avg=101.67
window=[99, 105, 110] -> moving_avg=104.67
window=[105, 110, 108] -> moving_avg=107.67
window=[110, 108, 112] -> moving_avg=110.00


## 18. Memory & Performance: List vs Generator at Scale

A quick, concrete demonstration of why this all matters: at real dataset sizes, the memory gap between an eagerly-built list and a lazy generator is enormous.

In [32]:
import sys

N = 1_000_000

big_list = [i for i in range(N)]
big_gen  = (i for i in range(N))

list_bytes = sys.getsizeof(big_list)
gen_bytes = sys.getsizeof(big_gen)

print(f"List of {N:,} ints:      {list_bytes:,} bytes")
print(f"Generator (same range): {gen_bytes:,} bytes")
print(f"The list uses ~{list_bytes / gen_bytes:,.0f}x more memory than the generator")

del big_list   # free the memory back up

List of 1,000,000 ints:      8,448,728 bytes
Generator (same range): 192 bytes
The list uses ~44,004x more memory than the generator


## 19. Pitfalls & Best Practices

- **Iterators (and generators) are single-pass.** Once exhausted, you can't loop over the same iterator object again — you must create a fresh one (e.g. call the generator function again, or `iter(obj)` again). A very common bug is reusing an already-exhausted iterator and getting silently empty results.
- **A generator function's body doesn't run at all until you start consuming it.** Calling `square(5)` only creates the generator object — nothing inside the function executes until the first `next()` (or the start of a `for` loop).
- **PEP 479 (Python 3.7+):** a bare `StopIteration` raised *accidentally* inside a generator's body is converted into a `RuntimeError`, specifically so it can't silently terminate an enclosing `for` loop early. Use `return` to end a generator early, not a manually raised `StopIteration`.
- **Priming is required before the first real `send()`.** As in the running-average example, you must call `next()` once to advance the generator to its first `yield` before you can usefully `.send()` a value into it.
- **Generators can't be indexed or measured with `len()`.** `gen[0]` and `len(gen)` both fail — convert to a list first only if you actually need that and it comfortably fits in memory.
- **Use `itertools.islice` to "preview" a generator**, since slicing syntax (`gen[:5]`) doesn't work on it directly.
- **Debugging a long generator pipeline is harder** than debugging an equivalent list-based one, because intermediate values are never materialized. When stuck, temporarily wrap one stage in `list(...)` to inspect it, then remove the wrapping again.
- **Multi-process data loading (e.g. `DataLoader(num_workers=...)`)** gives each worker process its *own copy* of a generator-based dataset — be deliberate about how file handles, random seeds, and any other state are shared or re-initialized across those processes.

## 20. Cheat Sheet

| Concept | Definition | Typical AI/ML Use Case |
|---|---|---|
| Iterable | Implements `__iter__` | Any dataset/collection |
| Iterator | Implements `__iter__` + `__next__`, stateful | Custom dataset iterator |
| `iter()` / `next()` | Get / advance an iterator | Manual control over a data stream |
| Generator function (`yield`) | Easiest way to build an iterator | Lazy preprocessing pipelines |
| Generator expression | Lazy version of a list comprehension | Memory-light transforms over big data |
| `send()` | Push a value into a paused generator | Online running-stat trackers (loss/accuracy) |
| `throw()` / `close()` | Inject an exception / stop early | Resource cleanup mid-stream |
| `yield from` | Delegate to a sub-generator | Flattening sharded datasets |
| `itertools.count/cycle/islice/chain` | Infinite & combining iterators | Step counters, device round-robin, bounded previews |
| `IterableDataset`-style class | `__iter__` returns a generator | Streaming datasets too large for RAM |
| Sliding-window generator | Lazy overlapping chunks | N-grams, time-series windows |

### Further Reading
- PEP 234 — Iterators
- PEP 255 — Simple Generators
- PEP 342 — Coroutines via Enhanced Generators (`send`/`throw`)
- PEP 380 — Syntax for Delegating to a Subgenerator (`yield from`)
- PEP 479 — Change `StopIteration` Handling Inside Generators
- The `itertools` module documentation
- PyTorch `Dataset` / `IterableDataset` documentation

This covers iterators and generators end-to-end — from the raw protocol that makes `for` loops work, through every generator feature, up to the exact streaming, batching, and windowing patterns you'll find inside real ML data pipelines.